# Active Space & Symmetry Reduction

Qubit reduction by Z2 symmetry -- the same idea as active-space selection for larger molecules.

**Objectives:**
- Diagonalize the full four-qubit H2 Hamiltonian for its ground energy
- Build the Z2-tapered single-qubit H2 Hamiltonian and confirm the same ground state
- See the qubit count collapse from 4 to 1 without changing the physics
- Connect this to active-space selection for larger molecules

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt
from lib.utils.statevector import statevector
np.random.seed(0)
device = LocalSimulator()

In [ ]:
# --- Chemistry kit (auto-provided; real STO-3G H2 data + exact helpers) ---
# H2 Jordan-Wigner qubit Hamiltonian at R = 0.75 Angstrom (STO-3G, big-endian:
# character k of each Pauli string acts on qubit k, qubit 0 = leftmost = MSB).
# Coefficients are REAL precomputed values (PennyLane differentiable Hartree-Fock).
H2_TERMS = [
    ("IIII", -0.1097305),
    ("IIIZ", -0.21886309),
    ("IIZI", -0.21886309),
    ("IIZZ", 0.17395379),
    ("IZII", 0.16988453),
    ("IZIZ", 0.12005143),
    ("IZZI", 0.16549432),
    ("XXYY", -0.04544288),
    ("XYYX", 0.04544288),
    ("YXXY", 0.04544288),
    ("YYXX", -0.04544288),
    ("ZIII", 0.16988453),
    ("ZIIZ", 0.16549432),
    ("ZIZI", 0.12005143),
    ("ZZII", 0.16821199),
]
H2_FCI = -1.137117   # exact ground-state energy (lowest eigenvalue), Hartree
H2_HF = -1.116151    # Hartree-Fock energy <1100|H|1100>, Hartree
# Z2 symmetry-tapered single-qubit form: H = c0*I + cz*Z + cx*X (ground = c0 - hypot(cz, cx))
H2_C0, H2_CZ, H2_CX = -0.338656, 0.777495, 0.181772

_PAULI = {
    "I": np.array([[1, 0], [0, 1]], dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def pauli_matrix(pauli_string):
    """Dense matrix of a big-endian Pauli string (qubit 0 = leftmost factor)."""
    m = np.array([[1.0 + 0j]])
    for ch in pauli_string:
        m = np.kron(m, _PAULI[ch])
    return m

def hamiltonian_matrix(terms):
    """Dense Hamiltonian sum(coeff * Pauli) from a list of (pauli_string, coeff)."""
    n = len(terms[0][0])
    h = np.zeros((2 ** n, 2 ** n), dtype=complex)
    for s, c in terms:
        h += c * pauli_matrix(s)
    return h

def hamiltonian_energy(state_vector, terms):
    """Expectation <psi|H|psi> for a qcsim state vector (real, in Hartree)."""
    h = hamiltonian_matrix(terms)
    return float(np.real(np.vdot(state_vector, h @ state_vector)))

def h2_double_excitation(theta):
    """HF |1100> plus the single Givens double excitation |1100> <-> |0011>.
    At the optimal theta this ansatz reaches the exact H2 ground state."""
    c = Circuit()
    c.x(0); c.x(1)
    c.cnot(2, 3); c.cnot(0, 2); c.h(0); c.h(3); c.cnot(0, 1); c.cnot(2, 3)
    c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(0, 3); c.h(3); c.cnot(3, 1); c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(2, 1); c.cnot(2, 0); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(3, 1); c.h(3); c.cnot(0, 3); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(0, 1); c.cnot(2, 0); c.h(0); c.h(3); c.cnot(0, 2); c.cnot(2, 3)
    return c
# --- end chemistry kit ---

## 1. The full four-qubit H2 Hamiltonian

`H2_TERMS` is the exact STO-3G H2 problem (R = 0.75 A): fifteen big-endian Pauli strings on **four qubits**. Its ground energy is the lowest eigenvalue; the kit gives `hamiltonian_matrix` and `H2_FCI` = -1.137117 Ha.

In [ ]:
H_full = hamiltonian_matrix(H2_TERMS)
n_qubits_full = len(H2_TERMS[0][0])
E_full_ground = np.linalg.eigvalsh(H_full)[0]
print(f"{len(H2_TERMS)} terms, {n_qubits_full} qubits | lowest {E_full_ground:.6f} | FCI {H2_FCI:.6f}")
assert np.isclose(E_full_ground, H2_FCI, atol=1e-4)
print("OK: four-qubit Hamiltonian gives the exact FCI ground energy.")

## 2. The same molecule on a single qubit

Each conserved Z2 symmetry tapers away one qubit: `H_tapered = c0*I + cz*Z + cx*X` (kit `H2_C0`, `H2_CZ`, `H2_CX`). Eigenvalues `c0 +/- sqrt(cz^2 + cx^2)`, ground energy `c0 - hypot(cz, cx)`.

In [ ]:
I2 = np.array([[1.0, 0.0], [0.0, 1.0]], dtype=complex)
Z2 = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)
X2 = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
H_tapered = H2_C0 * I2 + H2_CZ * Z2 + H2_CX * X2
n_qubits_tapered = 1
E_tapered_ground = np.linalg.eigvalsh(H_tapered)[0]
E_closed_form = H2_C0 - np.hypot(H2_CZ, H2_CX)
print(f"lowest {E_tapered_ground:.6f} | c0-hypot {E_closed_form:.6f} | FCI {H2_FCI:.6f}")
assert np.isclose(E_tapered_ground, E_closed_form, atol=1e-12)
assert np.isclose(E_tapered_ground, H2_FCI, atol=1e-4)
print("OK: one-qubit tapered Hamiltonian gives the exact FCI ground energy.")

## 3. Same ground state, a quarter of the qubits

The four-qubit and one-qubit operators agree on the lowest eigenvalue and both equal `H2_FCI`. Only the qubit budget changed.

In [ ]:
assert np.isclose(E_full_ground, E_tapered_ground, atol=1e-4)
qubit_reduction = n_qubits_full - n_qubits_tapered
print(f"qubits {n_qubits_full} -> {n_qubits_tapered} ({qubit_reduction} removed) | dim {2**n_qubits_full} -> {2**n_qubits_tapered}")
print("OK: 4 -> 1 qubit reduction, ground state preserved.")

## 4. Reaching the ground state

The one-qubit ground state comes from `eigh`; the four-qubit one is prepared via the kit's `h2_double_excitation(theta)`, reading energy from `statevector(...)` through `hamiltonian_energy`. The state vector is big-endian and spans the full four-qubit register. (Exercises in `../GUIDE.md`: tapered excited state c0+hypot, spectrum overlap, tapered HF vs FCI, single-qubit Ry VQE.)

In [ ]:
vecs = np.linalg.eigh(H_tapered)[1]
E_from_vec = np.real(vecs[:, 0].conj() @ H_tapered @ vecs[:, 0])
assert np.isclose(E_from_vec, H2_FCI, atol=1e-4)
thetas = np.linspace(-np.pi, np.pi, 401)
energies = np.array([hamiltonian_energy(statevector(h2_double_excitation(t)), H2_TERMS) for t in thetas])
best = int(np.argmin(energies))
print(f"best theta {thetas[best]:+.4f} -> {energies[best]:.6f} Ha (FCI {H2_FCI:.6f})")
assert np.isclose(energies[best], H2_FCI, atol=1e-3)
sv = statevector(h2_double_excitation(thetas[best]))
print(f"state vector length = {len(sv)} (= 2**4)")
print("OK: both representations reach the same -1.137 Ha ground state.")

## 5. Scaling up: this is active-space selection

Active-space selection drops uncorrelated orbitals: cheap **Hartree-Fock**, **keep orbitals near the Fermi level**, **freeze the rest**. The classical validator is **PySCF CASCI** (`mcscf.CASCI(mf, ncas=4, nelecas=4).kernel()` -- not run here, PySCF is browser-unavailable). H2O in STO-3G is a **14-qubit** problem cut to **4-to-8 qubits**. The naive qubit count is almost never the real one.

## Summary

- `H2_TERMS`: fifteen Pauli strings on **four qubits**, lowest eigenvalue the exact `H2_FCI` = -1.137 Ha.
- Z2-tapered it is `H = c0*I + cz*Z + cx*X` on **one qubit**; ground energy `c0 - hypot(cz, cx)` = `H2_FCI`.
- Both describe the identical ground state: **4 qubits collapse to 1**, physics untouched.
- Active-space selection in miniature; PySCF CASCI validates the choice and turns H2O from 14 qubits into 4-to-8.

**Next:** [`07-excited-states.ipynb`](07-excited-states.ipynb) -- subspace-search VQE for H2's first excited state.